# 04 EcoTrust Adaptive Embedding

Trains the adaptive latent encoder and saves 64-dimensional vectors for later FL experiments.

In [1]:
from pathlib import Path
import os
import sys

ABS_PROJECT_ROOT = Path('/Volumes/LocalDrive1/onlyscholar-projects/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')


def find_project_root():
    candidates = []
    env_root = os.environ.get('ECOTRUST_PROJECT_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([Path.cwd(), Path.cwd().parent, DRIVE_PROJECT_ROOT, ABS_PROJECT_ROOT])
    for base in list(candidates):
        candidates.extend(base.parents)
    seen = set()
    for candidate in candidates:
        expanded = candidate.expanduser()
        resolved = expanded.resolve() if expanded.exists() else expanded
        if str(resolved) in seen:
            continue
        seen.add(str(resolved))
        if (resolved / 'configs' / 'project.json').exists() and (resolved / 'src' / 'ecotrust').exists():
            return resolved
    raise RuntimeError(
        'Cannot find EcoTrust-FLIDS IMPLEMENTATION. In Colab, mount Drive and set ECOTRUST_PROJECT_ROOT to '
        '/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION.'
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if str(PROJECT_ROOT).startswith('/content/drive') and (PROJECT_ROOT / 'configs' / 'project_colab.json').exists():
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project_colab.json'
else:
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project.json'
RUN_MODE = 'full'  # change to 'full' after smoke run passes
DATASETS = ['Edge-IIoTset', 'TON_IoT', 'CICIoT2023']
print('PROJECT_ROOT:', PROJECT_ROOT)
print('CONFIG_PATH:', CONFIG_PATH)

import pandas as pd
from ecotrust.config import load_config
from ecotrust.io import write_json
from ecotrust.data.preprocess import load_preprocessed
from ecotrust.models.embedding import EmbeddingTrainConfig, train_embedding_model, encode_features
cfg = load_config(CONFIG_PATH)
DEVICE = 'cpu'
EPOCHS = 3 if RUN_MODE == 'smoke' else 10
for dataset in DATASETS:
    print('Embedding', dataset)
    prep = cfg.results_root / dataset / 'preprocessed' / RUN_MODE
    X_train, X_test, y_train, y_test = load_preprocessed(prep)
    out = cfg.results_root / dataset / 'embedding' / RUN_MODE
    model, history = train_embedding_model(X_train, y_train, out, EmbeddingTrainConfig(input_dim=X_train.shape[1], epochs=EPOCHS, device=DEVICE))
    pd.DataFrame(encode_features(model, X_train, DEVICE)).to_parquet(out / 'Z_train.parquet', index=False)
    pd.DataFrame(encode_features(model, X_test, DEVICE)).to_parquet(out / 'Z_test.parquet', index=False)
    write_json(out / 'embedding_history.json', history)
    print(history[-1])

PROJECT_ROOT: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION
CONFIG_PATH: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/configs/project_colab.json
Embedding Edge-IIoTset
{'epoch': 10, 'loss': 0.017539692307020925, 'accuracy': 1.0}
Embedding TON_IoT
{'epoch': 10, 'loss': 0.060202331315847186, 'accuracy': 0.9798717038017103}
Embedding CICIoT2023
{'epoch': 10, 'loss': 0.033796095099449155, 'accuracy': 0.99228}
